# Modelos de prioridad

El objetivo es decidir cuál de los cuatro predice mejor la prioridad de cada tramo: logística, árbol, kNN o SVM.

Al final medimos ese modelo una sola vez, en filas que no usamos para elegirlo.


## Datos

Días sin recolección, contenedores, quejas y tipo de zona entran porque con eso se armó la etiqueta. `cerca_escuela_hospital` y `feria_o_evento` no están en esa fórmula: las sumamos como contexto del tramo.

El id, la calle, la distancia y el tipo de vía quedan afuera. No dicen qué tan urgente es recolectar.

No hay nulos, así que no completamos ningún valor.


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "tramos.csv").exists():
    ROOT = ROOT.parent

tramos = pd.read_csv(ROOT / "data" / "processed" / "tramos.csv")

TARGET = "prioridad_recoleccion"
COLUMNAS = [
    "dias_desde_ultima_recoleccion",
    "cantidad_contenedores",
    "historico_quejas_vecinales",
    "cerca_escuela_hospital",
    "feria_o_evento",
    "tipo_zona",
]
NUMEROS = COLUMNAS[:-1]
ZONA = ["tipo_zona"]

X = tramos[COLUMNAS].copy()
y = tramos[TARGET].copy()

if X.isna().any().any():
    raise ValueError("Hay nulos en las columnas del modelo y no imputamos.")

tramos.shape

Out[0]: (1257, 13)


(1257, 13)

## Reparto

Apartamos un 20 % para test y no lo usamos para elegir. Si eligiéramos mirando esas filas, el resultado quedaría mejor de lo que el modelo rinde de verdad.

El otro 80 % se parte en cinco. Cada combinación se entrena cinco veces, y cada vez valida con una parte distinta. El número para comparar es el promedio del F1 macro. El desvío muestra cuánto se mueve ese promedio entre las cinco partes.

Un solo corte de validación, con este tamaño de datos, depende de qué filas tocaron. Por eso no usamos un 20 % fijo para decidir.

`stratify` mantiene la proporción de alta, media y baja en cada parte. `random_state=42` deja el mismo corte cada vez que se corre.

El ganador se vuelve a entrenar con todo el 80 %. El test se mira después, una sola vez.


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

pd.DataFrame(
    {
        "conjunto": ["train", "test"],
        "filas": [len(X_train), len(X_test)],
    }
)

Out[0]: 
  conjunto  filas
0    train   1005
1     test    252


,conjunto,filas
0,train,1005
1,test,252


## Qué se prueba

Comparamos los cuatro y nos quedamos con el de mayor F1 macro.

La baseline predice siempre la clase más frecuente, baja. No compite: sirve para ver si el modelo hace algo más que repetir esa mayoría.

Elegimos con F1 macro. Hay tres clases y baja es el 45 %: el accuracy premia acertar casi siempre baja y puede esconder que alta o media salen mal.

El árbol varía la profundidad y el mínimo de hojas, y no se escala: no mide distancias. kNN varía la cantidad de vecinos, solo los impares, para no dejar empates. SVM varía qué tan estricto es el margen y si la frontera es recta o curva. La logística no se grilla: dejamos el ajuste por defecto, con más iteraciones para que termine.

Logística, kNN y SVM escalan las cinco columnas numéricas. Si no, la de números más grandes pesa más solo por la escala. `tipo_zona` pasa a columnas de ceros y unos. Escalar y codificar se ajustan solo con las filas de train de cada ronda, para no usar las filas con las que después medimos.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


def preparar(escalar: bool) -> ColumnTransformer:
    numeros = StandardScaler() if escalar else "passthrough"
    return ColumnTransformer(
        [
            ("numeros", numeros, NUMEROS),
            ("zona", OneHotEncoder(handle_unknown="ignore"), ZONA),
        ]
    )


candidatos = {
    "Logística": (
        Pipeline(
            [
                ("prep", preparar(True)),
                ("modelo", LogisticRegression(max_iter=5000)),
            ]
        ),
        {},
    ),
    "Árbol": (
        Pipeline(
            [
                ("prep", preparar(False)),
                ("modelo", DecisionTreeClassifier(random_state=0)),
            ]
        ),
        {
            "modelo__max_depth": [2, 3, 4, 5, 8],
            "modelo__min_samples_leaf": [1, 5, 10],
        },
    ),
    "kNN": (
        Pipeline(
            [
                ("prep", preparar(True)),
                ("modelo", KNeighborsClassifier()),
            ]
        ),
        {"modelo__n_neighbors": list(range(1, 60, 2))},
    ),
    "SVM": (
        Pipeline(
            [
                ("prep", preparar(True)),
                ("modelo", SVC()),
            ]
        ),
        {
            "modelo__C": [0.01, 1, 100],
            "modelo__kernel": ["linear", "rbf"],
        },
    ),
}

busquedas = {}
filas = []
for nombre, (estimador, grilla) in candidatos.items():
    busqueda = GridSearchCV(
        estimador,
        grilla,
        scoring="f1_macro",
        cv=5,
    )
    busqueda.fit(X_train, y_train)
    busquedas[nombre] = busqueda
    mejor = busqueda.best_index_
    params = {
        clave.removeprefix("modelo__"): valor
        for clave, valor in busqueda.best_params_.items()
    }
    filas.append(
        {
            "modelo": nombre,
            "parametros": params or "max_iter=5000",
            "f1_macro": busqueda.cv_results_["mean_test_score"][mejor],
            "desvio": busqueda.cv_results_["std_test_score"][mejor],
        }
    )

baseline_scores = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro",
)
filas.append(
    {
        "modelo": "Baseline (siempre la más frecuente)",
        "parametros": "—",
        "f1_macro": baseline_scores.mean(),
        "desvio": baseline_scores.std(),
    }
)

tabla = pd.DataFrame(filas).sort_values("f1_macro", ascending=False, ignore_index=True)
tabla["f1_macro"] = tabla["f1_macro"].round(3)
tabla["desvio"] = tabla["desvio"].round(3)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
tabla

Out[0]: 
                                modelo                               parametros  f1_macro  desvio
0                                  SVM                {'C': 1, 'kernel': 'rbf'}     0.833   0.019
1                            Logística                            max_iter=5000     0.825   0.027
2                                Árbol  {'max_depth': 8, 'min_samples_leaf': 5}     0.823   0.026
3                                  kNN                       {'n_neighbors': 5}     0.785   0.027
4  Baseline (siempre la más frecuente)                                        —     0.207   0.001


,modelo,parametros,f1_macro,desvio
0,SVM,"{'C': 1, 'kernel': 'rbf'}",0.833,0.019
1,Logística,max_iter=5000,0.825,0.027
2,Árbol,"{'max_depth': 8, 'min_samples_leaf': 5}",0.823,0.026
3,kNN,{'n_neighbors': 5},0.785,0.027
4,Baseline (siempre la más frecuente),—,0.207,0.001


## Resultado

Gana el SVM, con `C=1` y kernel `rbf`. Es el F1 macro más alto de los cuatro: 0,833.


## Test

Estas filas no entraron en la elección. Medimos el SVM y la baseline en el mismo corte.


In [4]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

ETIQUETAS = ["alta", "media", "baja"]

pred = busquedas["SVM"].predict(X_test)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

pd.DataFrame(
    {
        "SVM": [
            accuracy_score(y_test, pred),
            f1_score(y_test, pred, average="macro"),
        ],
        "baseline": [
            accuracy_score(y_test, pred_baseline),
            f1_score(y_test, pred_baseline, average="macro"),
        ],
    },
    index=["accuracy", "f1_macro"],
).round(3)


Out[0]: 
            SVM  baseline
accuracy  0.829     0.452
f1_macro  0.821     0.208


,SVM,baseline
accuracy,0.829,0.452
f1_macro,0.821,0.208


In [5]:
pd.DataFrame(
    classification_report(y_test, pred, labels=ETIQUETAS, output_dict=True)
).round(3)


Out[0]: 
             alta   media     baja  accuracy  macro avg  weighted avg
precision   0.830   0.737    0.909     0.829      0.825         0.833
recall      0.780   0.795    0.877     0.829      0.818         0.829
f1-score    0.804   0.765    0.893     0.829      0.821         0.831
support    50.000  88.000  114.000     0.829    252.000       252.000


,alta,media,baja,accuracy,macro avg,weighted avg
precision,0.830,0.737,0.909,0.829,0.825,0.833
recall,0.780,0.795,0.877,0.829,0.818,0.829
f1-score,0.804,0.765,0.893,0.829,0.821,0.831
support,50.000,88.000,114.000,0.829,252.000,252.000


In [6]:
pd.DataFrame(
    confusion_matrix(y_test, pred, labels=ETIQUETAS),
    index=[f"real {etiqueta}" for etiqueta in ETIQUETAS],
    columns=[f"predicha {etiqueta}" for etiqueta in ETIQUETAS],
)


Out[0]: 
            predicha alta  predicha media  predicha baja
real alta              39              11              0
real media              8              70             10
real baja               0              14            100


,predicha alta,predicha media,predicha baja
real alta,39,11,0
real media,8,70,10
real baja,0,14,100


En el test el SVM llega a 0,829 de accuracy y 0,821 de F1 macro. La baseline, siempre baja, queda en 0,452 y 0,208.

De 50 tramos altos acierta 39 y los otros 11 los marca como media. De 88 medios acierta 70; 8 quedan como alta y 10 como baja. De 114 bajos acierta 100 y los otros 14 quedan como media. Alta y baja no se cruzan.
